# 14. User Movement Evaluation

Run the implemented ①-⑪ Python pipeline on one pose CSV. The default sample is a p01 squat recording. Edit only the input cell to evaluate another movement recording.

**Prerequisite:** install the package in editable mode and confirm `setup_00_environment_check.ipynb` passes.

**Input:** MediaPipe-style pose CSV, optional annotation CSV, and optional de-identified participant YAML.

**Output:** processed dataframe, pipeline report, feature records, biomechanical proxy records, and biomarker scores under `data/processed/user_movement_evaluation/` when saving is enabled.

**Boundary:** corrected-3D-hypothesis depth remains canonicalization candidate evidence only; score gravity is deferred to a later scoring-policy task.


In [ ]:
%load_ext autoreload
%autoreload 2


## Input Settings


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

POSE_CSV = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"
ANNOTATION_CSV = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv"
PARTICIPANT_PROFILE_YAML = PROJECT_ROOT / "data/participants/no_consent/p01.yaml"
EXERCISE_ID = "squat"
DEFINITIONS_DIR = PROJECT_ROOT / "data/definitions/exercises"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/user_movement_evaluation"
SAVE_OUTPUTS = True

print("project root:", PROJECT_ROOT)
print("pose csv:", POSE_CSV.relative_to(PROJECT_ROOT))
print("annotation csv:", ANNOTATION_CSV.relative_to(PROJECT_ROOT) if ANNOTATION_CSV.exists() else "not found")
print("participant profile:", PARTICIPANT_PROFILE_YAML.relative_to(PROJECT_ROOT) if PARTICIPANT_PROFILE_YAML.exists() else "not found")


## Load Data


In [ ]:
import json

import pandas as pd

from movement.annotation import load_annotation_csv
from movement.config import LANDMARKS
from movement.io import load_participant_profile_yaml, load_pose_csv
from movement.pipeline import (
    AnnotationConfig,
    BiomechConfig,
    BiomarkerConfig,
    ExerciseDefinitionConfig,
    FeaturesConfig,
    MotionAttributionConfig,
    NormalizationConfig,
    PhaseSegmentationConfig,
    PipelineConfig,
    PreprocessingConfig,
    ValidationConfig,
    run_pipeline,
)

pose_df = load_pose_csv(POSE_CSV)
annotation_df = load_annotation_csv(ANNOTATION_CSV) if ANNOTATION_CSV.exists() else None
participant_profile = (
    load_participant_profile_yaml(PARTICIPANT_PROFILE_YAML)
    if PARTICIPANT_PROFILE_YAML.exists()
    else None
)

input_summary = {
    "pose_frames": len(pose_df),
    "pose_columns": len(pose_df.columns),
    "annotation_rows": 0 if annotation_df is None else len(annotation_df),
    "exercise_id": EXERCISE_ID,
    "participant_id": None if participant_profile is None else participant_profile.get("participant_id"),
    "participant_sex": None if participant_profile is None else participant_profile.get("anthropometry", {}).get("sex"),
    "height_bin": None if participant_profile is None else participant_profile.get("anthropometry", {}).get("height_bin"),
    "common_subject_skeleton_profile": None if participant_profile is None else participant_profile.get("common_subject_skeleton", {}).get("profile_id"),
    "participant_profile_used_for_scoring": None if participant_profile is None else participant_profile.get("policy", {}).get("used_for_scoring"),
}
display(pd.DataFrame([input_summary]))


## Run Pipeline


In [ ]:
has_annotation = annotation_df is not None

config = PipelineConfig()
config.validation = ValidationConfig(enabled=True)
config.annotation = AnnotationConfig(
    enabled=has_annotation,
    path=str(ANNOTATION_CSV) if has_annotation else None,
)
config.exercise_definition = ExerciseDefinitionConfig(
    enabled=True,
    definitions_dir=str(DEFINITIONS_DIR),
    exercise_id=EXERCISE_ID,
)
config.preprocessing = PreprocessingConfig(enabled=True)
config.normalization = NormalizationConfig(enabled=True)
config.phase_segmentation = PhaseSegmentationConfig(
    enabled=has_annotation,
    fps_default=30.0,
)
config.motion_attribution = MotionAttributionConfig(enabled=True)
config.features = FeaturesConfig(enabled=True)
config.biomech = BiomechConfig(enabled=True)
config.biomarker = BiomarkerConfig(enabled=True)

result_df, report = run_pipeline(
    pose_df,
    config,
    ann_df=annotation_df,
    landmarks=LANDMARKS,
)

step_summary = {
    "validation_passed": report.get("validation", {}).get("passed"),
    "frames_after_pipeline": len(result_df),
    "normalization_depth_scale": report.get("normalization", {}).get("model_depth_scale"),
    "corrected_depth_used_downstream": report.get("canonicalization", {})
    .get("corrected_3d_hypothesis", {})
    .get("used_for_features_or_scores"),
    "feature_records": len(report.get("features", [])),
    "biomech_records": len(report.get("biomech", [])),
    "score_records": len(report.get("biomarker_scores", [])),
}
display(pd.DataFrame([step_summary]))


## Review Feature And Score Tables


In [ ]:
features_df = pd.DataFrame(report.get("features", []))
biomech_df = pd.DataFrame(report.get("biomech", []))
scores_df = pd.DataFrame(report.get("biomarker_scores", []))

if features_df.empty:
    print("No feature records were emitted. Check annotation/rep labels and exercise definition.")
else:
    display(features_df.head(12))

if biomech_df.empty:
    print("No biomechanical proxy records were emitted.")
else:
    display(biomech_df.head(12))

if scores_df.empty:
    print("No biomarker scores were emitted. Rep annotation is usually required for scoring.")
else:
    display(scores_df)


## Save Outputs


In [ ]:
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    processed_path = OUTPUT_DIR / "processed_pose.csv"
    report_path = OUTPUT_DIR / "pipeline_report.json"
    scores_path = OUTPUT_DIR / "biomarker_scores.csv"

    result_df.to_csv(processed_path, index=False)
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False, default=str)
    if not scores_df.empty:
        scores_df.to_csv(scores_path, index=False)

    print("saved:", processed_path.relative_to(PROJECT_ROOT))
    print("saved:", report_path.relative_to(PROJECT_ROOT))
    if not scores_df.empty:
        print("saved:", scores_path.relative_to(PROJECT_ROOT))
else:
    print("SAVE_OUTPUTS is False; no files written.")
